# NN-Assisted Rate Theory — Irradiation Damage in 316 SS

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import time, warnings
warnings.filterwarnings('ignore')

try:
    import joblib
    try:
        model  = joblib.load("nn_model.pkl")
        scaler = joblib.load("nn_scaler.pkl")
        print("Model:", model.hidden_layer_sizes)
        USE_NN = True
    except FileNotFoundError:
        print("nn_model.pkl / nn_scaler.pkl not found — using heuristic x_max gating")
        model  = None
        scaler = None
        USE_NN = False
except ImportError:
    print("joblib not available — using heuristic x_max gating")
    model  = None
    scaler = None
    USE_NN = False

In [ ]:
params = {
    'C_i1':84, 'E_m_v':1.6, 'E_m_i':0.2, 'E_f_v':1.6, 'E_f_i':4.08,
    'E_m_2v':0.9, 'E_b_2v':0.25, 'a_A':3.63,
    'nu_i':5e12, 'nu_v':5e13, 'Z_v':1.0, 'Z_i':1.08, 'g':6.24e14,
}
kB=8.617333262e-5; T=723.15; rho_d=1e9; P=1e-6; E_b_2i=1.19
a=params['a_A']*1e-8

Di  = params['nu_i']*a**2*np.exp(-params['E_m_i']/(kB*T))
Dv  = params['nu_v']*a**2*np.exp(-params['E_m_v']/(kB*T))
D2v = params['nu_v']*a**2*np.exp(-params['E_m_2v']/(kB*T))
alpha   = 48*params['nu_i']*np.exp(-params['E_m_i']/(kB*T))
Cv_eq   = np.exp(-params['E_f_v']/(kB*T))
C2v_eq  = 6*np.exp(-(2*params['E_f_v']-params['E_b_2v'])/(kB*T))
K_nuc_i = params['C_i1']*params['nu_i']*np.exp(-params['E_m_i']/(kB*T))

Nv, Ni = 50, 100
N = Nv + Ni
print(f"Cv_eq={Cv_eq:.3e}  alpha={alpha:.3e}")

In [ ]:
def _Kc_v(x): return (2.216*params['Z_v']**2*x**(2/3)/(1+0.1128*params['Z_v']*x**(1/3))
                      *params['nu_v']*np.exp(-params['E_m_v']/(kB*T)))
def _Kc_i(x): return (2.216*params['Z_i']**2*x**(2/3)/(1+0.1128*params['Z_i']*x**(1/3))
                      *params['nu_i']*np.exp(-params['E_m_i']/(kB*T)))
def _Kl_v(x): return 1.555*params['Z_v']*x**0.5*params['nu_v']*np.exp(-params['E_m_v']/(kB*T))
def _Kl_i(x): return 1.555*params['Z_i']*x**0.5*params['nu_i']*np.exp(-params['E_m_i']/(kB*T))
def _gamma_cv(x): return _Kc_v(x)*Cv_eq*np.exp((1.28*params['g']*a**2/(kB*T))*x**(-1/3))
def _gamma_lv(x): return _Kl_v(2)*np.exp(-E_b_2i/(kB*T)) if int(x)==2 else 0.0

KCV = np.array([_Kc_v(x)    for x in range(1, Nv+1)])
KCI = np.array([_Kc_i(x)    for x in range(1, Nv+1)])
KLV = np.array([_Kl_v(x)    for x in range(1, Ni+1)])
KLI = np.array([_Kl_i(x)    for x in range(1, Ni+1)])
GCV = np.array([_gamma_cv(x) for x in range(1, Nv+1)])
GLV = np.array([_gamma_lv(x) for x in range(1, Ni+1)])
print("Rate constants cached.")

In [ ]:
def rhs_full(t, y):
    Cv_arr = np.maximum(y[:Nv], 1e-100)
    Ci_arr = np.maximum(y[Nv:], 1e-100)
    dydt   = np.zeros(N)
    Cv=Cv_arr[0]; Ci=Ci_arr[0]; C2v=Cv_arr[1]; C2i=Ci_arr[1]

    dCv = P + KCI[1]*Ci*C2v + (2*GCV[1]-KCV[1]*Cv)*C2v
    dCv += np.dot(GCV[2:Nv], Cv_arr[2:Nv])
    dCv -= np.dot(KCV[2:Nv], Cv_arr[2:Nv])*Cv
    dCv -= np.dot(KLV[2:Ni], Ci_arr[2:Ni])*Cv
    dCv += params['Z_v']*rho_d*Dv*(Cv_eq-Cv) - alpha*Cv*Ci - KCV[0]*Cv**2 - KLV[1]*Cv*C2i
    dydt[0] = dCv

    dC2v = 0.5*KCV[0]*Cv**2 + GCV[2]*Cv_arr[2] + KCI[2]*Ci*Cv_arr[2] + rho_d*D2v*(C2v_eq-C2v)
    dC2v -= (KCV[1]*Cv + KCI[1]*Ci + GCV[1])*C2v
    dydt[1] = dC2v

    for x in range(3, Nv+1):
        i=x-1; Cxm1=Cv_arr[x-2]; Cx=Cv_arr[x-1]
        dCx = KCV[x-2]*Cv*Cxm1 - KCI[x-1]*Ci*Cx - KCV[x-1]*Cv*Cx - GCV[x-1]*Cx
        if x < Nv: dCx += KCI[x]*Ci*Cv_arr[x] + GCV[x]*Cv_arr[x]
        dydt[i] = dCx

    dCi = P + KLV[1]*Cv*C2i - K_nuc_i*Ci**2 - alpha*Cv*Ci - KLI[1]*Ci*C2i - KCI[1]*Ci*C2v
    dCi -= np.dot(KLI[2:Ni], Ci_arr[2:Ni])*Ci
    dCi -= np.dot(KCV[2:Nv], Cv_arr[2:Nv])*Ci
    dCi -= params['Z_i']*rho_d*Di*Ci
    dydt[Nv] = dCi

    dC2i = 0.5*K_nuc_i*Ci**2 + KLV[2]*Cv*Ci_arr[2] - (KLI[1]*Ci + KLV[1]*Cv)*C2i
    dydt[Nv+1] = dC2i

    for x in range(3, Ni+1):
        i=Nv+x-1; Cxm1=Ci_arr[x-2]; Cx=Ci_arr[x-1]
        dCx = KLI[x-2]*Ci*Cxm1 + GLV[x-2]*Cxm1 - (KLI[x-1]*Ci + KLV[x-1]*Cv)*Cx
        if x < Ni: dCx += KLV[x]*Cv*Ci_arr[x]
        dydt[i] = dCx

    return dydt
print("rhs_full defined.")

## NN x_max Gating

**Key insight:** only the NN's `x_max` prediction is used to suppress sizes
that haven't nucleated yet (`dC/dt = 0` for sizes > x_max).

In [ ]:
def predict_xmax(t_now, Cv, Ci, x_max_now, x_min_now,
                 dlogCi=0.0, dose_rate=1e-6, margin=3):
    """Query NN for x_max, or use a time-based heuristic if model is unavailable."""
    if USE_NN:
        feat = np.array([[
            np.log10(max(t_now,    1e-30)),
            np.log10(max(Cv,       1e-100)),
            np.log10(max(Ci,       1e-100)),
            x_max_now / Ni,
            x_min_now / Ni,
        ]], dtype=np.float32)
        pred = model.predict(scaler.transform(feat))[0]
        x_max_pred = int(np.round(pred[0] * Ni)) + margin
    else:
        # Heuristic: grow x_max linearly on a log-time axis from t=1e-6 to t=1e5
        t_frac = np.clip(np.log10(max(t_now, 1e-6) / 1e-6) / 11.0, 0.0, 1.0)
        x_max_pred = int(3 + (Ni - 3) * t_frac) + margin
    # x_max never shrinks; capped at Ni
    return min(max(x_max_pred, x_max_now), Ni)

def make_xmax_rhs(x_max):
    """RHS that zeroes dC/dt for interstitial sizes > x_max (not yet nucleated)."""
    freeze_hi = slice(Nv + x_max, N)
    def gated(t, y, fh=freeze_hi):
        dydt = rhs_full(t, y)
        dydt[fh] = 0.0
        return dydt
    return gated


## NN-Assisted Integrator

In [ ]:
def integrate_rt_nn(y0, t_span, n_segments=60,
                    rtol=1e-8, atol=1e-50, dose_rate=1e-6, verbose=True):
    """
    Integrate rate-theory ODEs with NN-gated x_max upper cutoff.
    Only integrates sizes up to x_max per segment; sizes above are frozen at zero.

    Returns
    -------
    t_arr    : (n_segments+1,)
    y_arr    : (N, n_segments+1)
    xmax_arr : (n_segments,)  x_max used per segment
    """
    t_edges = np.logspace(np.log10(t_span[0]), np.log10(t_span[1]), n_segments+1)

    y_cur    = y0.copy()
    t_out    = [t_edges[0]]
    y_out    = [y0.copy()]
    xmax_history = []

    x_max_cur = 3
    x_min_cur = 1   # for NN feature only — not used for freezing
    dlogCi    = 0.0

    for seg in range(n_segments):
        t0, t1 = t_edges[seg], t_edges[seg+1]

        x_max_next = predict_xmax(
            t0, float(y_cur[0]), float(y_cur[Nv]),
            x_max_cur, x_min_cur, dlogCi=dlogCi, dose_rate=dose_rate, margin=3,
        )
        xmax_history.append(x_max_next)

        gated = make_xmax_rhs(x_max_next)
        sol   = solve_ivp(gated, (t0, t1), y_cur,
                          method='LSODA', rtol=rtol, atol=atol)

        if not sol.success and verbose:
            print(f"  [seg {seg}] {sol.message}")

        y_cur = np.maximum(sol.y[:, -1], 0.0)
        x_max_cur = x_max_next

        t_out.append(t1)
        y_out.append(y_cur.copy())

        if verbose and (seg % 10 == 0 or seg == n_segments-1):
            pct = 100 * x_max_next / Ni
            print(f"  seg {seg:3d}  t={t1:.2e}s  x_max={x_max_next:3d} ({pct:.0f}% of Ni)"
                  f"  Ci={float(y_cur[Nv]):.2e}")

    return np.array(t_out), np.column_stack(y_out), np.array(xmax_history)

print("integrate_rt_nn defined.")

In [ ]:
y0 = np.zeros(N)
y0[0]    = Cv_eq
y0[1]    = 1e-6 * Cv_eq
y0[2]    = 1e-8 * Cv_eq
y0[Nv]   = 1e-20
y0[Nv+1] = 1e-30
y0[Nv+2] = 1e-35

print("=" * 55)
print("NN-assisted RT integration")
print("=" * 55)
t0w = time.perf_counter()

t_nn, y_nn, xmax_arr = integrate_rt_nn(
    y0, t_span=(1e-6, 1e5),
    n_segments=60, rtol=1e-8, atol=1e-50,
    dose_rate=1e-6, verbose=True,
)
elapsed = time.perf_counter() - t0w
print(f"\n time: {elapsed:.1f} s")

## Plot — Defect Concentrations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))

ax.plot(t_nn, np.maximum(y_nn[0],  1e-100), 'k-',  lw=2.5, label=r'$C_v$')
ax.plot(t_nn, np.maximum(y_nn[Nv], 1e-100), 'k--', lw=2.5, label=r'$C_i$')

loop_sizes = [2, 3, 4, 6, 12, 18, 30, 50, 70]
colors = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple',
          'tab:brown','tab:pink','tab:gray','tab:olive']
for ls, col in zip(loop_sizes, colors):
    if ls > Ni: continue
    ax.plot(t_nn, np.maximum(y_nn[Nv+ls-1], 1e-100),
            lw=1.8, color=col, label=f'$C_{{{ls}i}}$')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim([1e-6, 1e5]); ax.set_ylim([1e-20, 1e-5])
ax.set_xlabel('t (s)', fontsize=13)
ax.set_ylabel('Defect concentration (at/at)', fontsize=13)
ax.set_title(r'$T=450\,°C$,  $\rho_d=10^9$ cm/cm$^3$,  dose rate $=10^{-6}$ dpa/s',
             fontsize=11)
ax.legend(ncol=2, fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Plot — Active Band vs Time

In [ ]:
# Derive x_min and x_max from concentration threshold for plotting
THRESH = 1e-18
x_max_plot, x_min_plot = [], []
for j in range(y_nn.shape[1]):
    appeared = np.where(y_nn[Nv:, j] > THRESH)[0]
    x_max_plot.append(int(appeared[-1]+1) if len(appeared) else 0)
    x_min_plot.append(int(appeared[0]+1)  if len(appeared) else 1)
x_max_plot = np.array(x_max_plot)
x_min_plot = np.array(x_min_plot)

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(t_nn, x_min_plot, x_max_plot,
                alpha=0.3, color='tab:blue', label='Active band')
ax.plot(t_nn, x_max_plot, 'b-',  lw=2,   label=r'$x_{\mathrm{max}}$')
ax.plot(t_nn, x_min_plot, 'b--', lw=1.5, label=r'$x_{\mathrm{min}}$')
ax.axhline(Ni, color='k', ls=':', lw=1.2, label=f'Full system ({Ni} sizes)')
ax.set_xscale('log')
ax.set_xlim([t_nn[1], t_nn[-1]])
ax.set_ylim([0, Ni+5])
ax.set_xlabel('t (s)', fontsize=13)
ax.set_ylabel('Loop size index', fontsize=13)
ax.set_title('Active loop size band vs time  —  NN-gated integration', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()